# Publish packed stores to Kaggle (Modal Notebook, **CPU**)

One Kaggle dataset per store directory on the `depthwizard-data` Volume.
`dfc23_g050` is already up, so it sits in `SKIP` below with `_dl/` (staging junk).

`kaggle` cannot upload a nested tree, so `--dir-mode zip` is used: each split is
zipped and uploaded. **Kaggle expands archives server-side**, so the dataset
mounts at `/kaggle/input/<slug>/train/index.json` with the tree intact -- no
unzip step in the consuming notebook, and `packed.py` opens the stores unchanged.

| Sidebar tab | Set to |
|---|---|
| Compute profile | **4 CPU, 16 GiB, no GPU** -- this is network and zip, not compute |
| Compute profile | **Idle timeout >= 60 min** (default is 10) |
| Filesystem | Volume `depthwizard-data` -> mounts at `/mnt/depthwizard-data` |
| Secrets (or cell 1) | `dw-kaggle` (`KAGGLE_USERNAME`, `KAGGLE_KEY`) |

The CLI builds each archive on local disk before sending a byte, so the
container needs free scratch space equal to the largest store. Cell 2 checks
that and refuses rather than dying halfway through a 50 GiB zip.

Upload is serial and slow. Re-running is safe: a store that already exists on
Kaggle gets a new *version* instead of erroring out.

In [ ]:
# 1. Credentials. LEAVE BLANK IN THE COMMITTED COPY - fill them in on your copy
#    inside Modal. Blank falls through to the dw-kaggle Secret from the sidebar.
import os

KAGGLE_USERNAME = ""      # kaggle.com -> Settings -> API -> Create New Token
KAGGLE_KEY      = ""      # ...both fields live in the kaggle.json it downloads

for k, v in {"KAGGLE_USERNAME": KAGGLE_USERNAME, "KAGGLE_KEY": KAGGLE_KEY}.items():
    if v:
        os.environ[k] = v
    elif not os.environ.get(k):
        raise SystemExit(f"[dw] {k} is unset: paste it above or attach the dw-kaggle Secret")
print("[dw] kaggle user:", os.environ["KAGGLE_USERNAME"])

In [ ]:
%uv pip install "kaggle>=1.6.0"

In [ ]:
# 2. What goes up, and whether there is room to stage it. Edit SKIP, not the loop.
import json, shlex, shutil, subprocess, time
from pathlib import Path

DATA   = Path("/mnt/depthwizard-data")
SKIP   = {"_dl", "dfc23_g050"}           # staging junk; already published
PUBLIC = False                           # True adds -u and the dataset is world-readable
USER   = os.environ["KAGGLE_USERNAME"]

# The CLI zips into tempfile.mkdtemp(). Left at the default that is the container
# root filesystem, which is small; /scratch is the big container-local disk.
STAGE = Path("/scratch/kagglezip")
STAGE.mkdir(parents=True, exist_ok=True)
os.environ["TMPDIR"] = str(STAGE)

STORES = [p for p in sorted(DATA.iterdir()) if p.is_dir() and p.name not in SKIP]

def gib(n):
    return n / 2**30

def size(p):
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())

sizes = {p: size(p) for p in STORES}
for p, n in sizes.items():
    print(f"{p.name:<16} {gib(n):6.1f} GiB   -> {USER}/depthwizard-{p.name.replace('_', '-')}")

free = shutil.disk_usage(STAGE).free
need = max(sizes.values(), default=0)
print(f"\nstaging {STAGE}: {gib(free):.0f} GiB free, largest store needs ~{gib(need):.0f} GiB")
print("skipped:", ", ".join(sorted(SKIP)))
assert free > need * 1.05, "not enough scratch to stage the largest store -- raise the disk or SKIP it"

In [ ]:
# 3. Upload, one store at a time. Slow and chatty on purpose - a silent hour is
#    indistinguishable from a hang.
#
# We build the archives ourselves rather than passing `-r zip`, because the CLI's
# dir-mode uses shutil.make_archive -> ZIP_DEFLATED on one thread: hours of a
# single pegged core, on float .npy shards that compress by almost nothing.
# ZIP_STORED is a copy, so this is disk-bound and the extra cores stop mattering.
# Kaggle expands both kinds server-side, so the mounted tree is identical.
import zipfile

def stage_zips(store: Path) -> Path:
    out = STAGE / store.name
    out.mkdir(parents=True, exist_ok=True)
    for split in sorted(d for d in store.iterdir() if d.is_dir()):
        z = out / f"{split.name}.zip"
        if z.exists():
            print(f"[dw]   {z.name} already staged", flush=True)
            continue
        t = time.time()
        tmp = z.with_suffix(".zip.part")     # never leave a truncated zip behind
        with zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED) as zf:
            for f in sorted(split.rglob("*")):
                if f.is_file():
                    zf.write(f, f"{split.name}/{f.relative_to(split)}")
        tmp.rename(z)
        print(f"[dw]   {z.name} {gib(z.stat().st_size):.1f} GiB "
              f"({(time.time() - t) / 60:.1f} min)", flush=True)
    return out


# Kaggle rejects a title under 6 chars, and an id must be lowercase [a-z0-9-].
def publish(store: Path):
    slug = f"depthwizard-{store.name.replace('_', '-')}"
    print(f"\n[dw] {store.name} -> {slug}", flush=True)
    folder = stage_zips(store)
    (folder / "dataset-metadata.json").write_text(json.dumps({
        "title": f"DepthWizard {store.name}",
        "id": f"{USER}/{slug}",
        "licenses": [{"name": "CC0-1.0"}],
    }, indent=2))

    base = ["kaggle", "datasets"]
    tail = ["-p", str(folder)]               # flat folder of .zip, so no -r needed
    exists = subprocess.run(base + ["status", f"{USER}/{slug}"],
                            capture_output=True, text=True).returncode == 0
    cmd = (base + ["version"] + tail + ["-m", f"packed shards {time.strftime('%Y-%m-%d')}"]
           if exists else
           base + ["create"] + tail + (["-u"] if PUBLIC else []))

    print(f"[dw] $ {shlex.join(cmd)}", flush=True)
    t = time.time()
    if subprocess.run(cmd, check=False).returncode:
        raise RuntimeError(f"[dw] {slug} failed after {(time.time() - t) / 60:.1f} min")
    print(f"[dw] {slug} ok ({(time.time() - t) / 60:.1f} min) "
          f"https://www.kaggle.com/datasets/{USER}/{slug}", flush=True)
    for z in folder.glob("*.zip"):           # scratch is not free, and the next
        z.unlink()                           # store needs the room


for s in STORES:
    publish(s)

Kaggle takes a few minutes per dataset after the upload returns to expand the
archives; until it does, the page shows the raw `.zip`. Check one dataset's file
list before pointing a training notebook at it.

A store that dies mid-upload leaves nothing half-published -- rerun cell 3. Zips
already staged are reused, and `status` finding no dataset sends it back down the
`create` path.